In [1]:
import sys
from pathlib import Path

# Đảm bảo project root nằm trong sys.path để import src.*
_current = Path.cwd().resolve()
PROJECT_ROOT = _current
while PROJECT_ROOT.name != "RecommendationSystem" and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

import numpy as np
import pandas as pd
from scipy.sparse import save_npz

from src.data_loader import ML25MLoader, DataPreprocessor, save_processed_data

print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("Project root:", PROJECT_ROOT)

Python: 3.13.14
numpy: 2.5.2
pandas: 3.0.5
Project root: /home/huyquang/Workspace/Personal/PyCharmProjects/RecommendationSystem


### 1. Load dữ liệu & EDA tối thiểu

Loader `ML25MLoader` ở `src/data_loader.py` : `load_ratings()` và `load_movies()`.

EDA tối thiểu in 4 số liệu cần cho báo cáo: tổng rating, số user, số phim, khoảng thời gian, tỉ lệ rating ≥ 4.

In [2]:
loader = ML25MLoader(data_dir=PROJECT_ROOT / "data/raw/ml-25m")
ratings = loader.load_ratings()
movies = loader.load_movies()

print(f"Total ratings : {len(ratings):,}")
print(f"Unique users  : {ratings['userId'].nunique():,}")
print(f"Unique movies : {ratings['movieId'].nunique():,}")
ts_min = pd.to_datetime(ratings["timestamp"], unit="s").min()
ts_max = pd.to_datetime(ratings["timestamp"], unit="s").max()
print(f"Time range    : {ts_min}  →  {ts_max}")
print(f"Rating ≥ 4    : {(ratings['rating'] >= 4).mean() * 100:.2f}%")

ratings.head()

Total ratings : 25,000,095
Unique users  : 162,541
Unique movies : 59,047
Time range    : 1995-01-09 11:46:49  →  2019-11-21 09:15:03
Rating ≥ 4    : 49.81%


/home/huyquang/miniconda3/envs/venv/lib/python3.13/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


### 2. Temporal split, cold-start filter, ID mapping

Bước này sinh ra **bốn artifact dùng chung** cho các phần sau:

| Artifact                 | Vai trò                                                                      |
|--------------------------|------------------------------------------------------------------------------|
| `train_raw`              | Toàn bộ explicit ratings cho train (dùng cho Popularity, User-CF, Item-CF).  |
| `test_warm`              | Test đã lọc cold-start (cùng metric ranking cho mọi mô hình).                |
| `explicit_user_item_csr` | Ma trận user × item với **giá trị rating** (cho Popularity/User-CF/Item-CF). |
| `implicit_user_item_csr` | Ma trận user × item với **confidence `1 + α · rating`** (cho Implicit ALS).  |

In [3]:
pre = DataPreprocessor()

# 1) Temporal split: 70% train / 30% test theo timestamp.
train_raw, test_raw = pre.temporal_split(ratings, train_ratio=0.7)
print(f"Train rows : {len(train_raw):,}")
print(f"Test rows  : {len(test_raw):,}")

# 2) Cold-start filter: giữ lại chỉ user/item đã thấy trong train.
test_warm = pre.filter_cold_start(train_raw, test_raw)

# 3) Ma trận explicit cho Popularity, User-CF và Item-CF.
explicit_user_item_csr, explicit_user_map, explicit_movie_map = (
    pre.build_user_item_matrix(train_raw, value_column="rating")
)

# 4) Ma trận implicit/confidence riêng cho ALS (rating >= 4).
train_implicit = pre.to_implicit(train_raw, threshold=4.0)
test_implicit = pre.to_implicit(test_warm, threshold=4.0)
implicit_user_item_csr, implicit_user_map, implicit_movie_map = (
    pre.build_user_item_matrix(train_implicit, confidence_alpha=1.0)
)

Train rows : 17,500,066
Test rows  : 7,500,029
Removed 6,951,670 cold-start interactions (92.69% of test)
User-item matrix: (122029, 22931), sparsity: 0.6254%, nnz: 17,500,066
Implicit conversion: 8,750,986 / 17,500,066 (50.0%) positive interactions
Implicit conversion: 212,129 / 548,359 (38.7%) positive interactions
User-item matrix: (121829, 18405), sparsity: 0.3903%, nnz: 8,750,986


In [5]:
# 5) Lưu artifact ra data/processed/ để các notebook khác (02-cf-experiments) tái sử dụng.
save_processed_data(train_raw, test_warm, save_dir=PROJECT_ROOT / "data/processed")

processed_dir = Path(PROJECT_ROOT / "data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)
save_npz(processed_dir / "train_implicit_user_item.npz", implicit_user_item_csr)

print("Artifacts saved:")
for p in sorted(processed_dir.iterdir()):
    print(f"  - {p}")

Saved to /home/huyquang/Workspace/Personal/PyCharmProjects/RecommendationSystem/data/processed:
  - train_ml25m.csv: 17,500,066 rows
  - test_ml25m.csv: 548,359 rows
Artifacts saved:
  - /home/huyquang/Workspace/Personal/PyCharmProjects/RecommendationSystem/data/processed/test_ml25m.csv
  - /home/huyquang/Workspace/Personal/PyCharmProjects/RecommendationSystem/data/processed/train_implicit_user_item.npz
  - /home/huyquang/Workspace/Personal/PyCharmProjects/RecommendationSystem/data/processed/train_ml25m.csv


Verify data processed results

In [8]:
from scipy.sparse import load_npz

processed_dir = Path(PROJECT_ROOT / "data/processed")

# 1) File outputs tồn tại.
train_csv = processed_dir / "train_ml25m.csv"
test_csv = processed_dir / "test_ml25m.csv"
implicit_npz = processed_dir / "train_implicit_user_item.npz"

assert train_csv.exists(), f"Missing {train_csv}"
assert test_csv.exists(), f"Missing {test_csv}"
assert implicit_npz.exists(), f"Missing {implicit_npz}"
print("[OK] All required files exist.")

# 2) Hai matrix tách biệt và có ID mapping riêng.
assert explicit_user_item_csr.shape[0] == len(explicit_user_map)
assert explicit_user_item_csr.shape[1] == len(explicit_movie_map)
assert implicit_user_item_csr.shape[0] == len(implicit_user_map)
assert implicit_user_item_csr.shape[1] == len(implicit_movie_map)
print("[OK] Two distinct matrices (explicit & implicit) with matching ID maps.")

# 3) Không còn cold-start user/item trong test_warm.
train_users = set(train_raw["userId"].unique())
train_movies = set(train_raw["movieId"].unique())
assert test_warm["userId"].isin(train_users).all()
assert test_warm["movieId"].isin(train_movies).all()
print("[OK] No cold-start users/items remain in test_warm.")

# 4) Sparsity implicit ~ 99.7%.
imp_nnz = implicit_user_item_csr.nnz
imp_total = implicit_user_item_csr.shape[0] * implicit_user_item_csr.shape[1]
imp_sparsity = 100 * (1 - imp_nnz / imp_total)
print(f"[INFO] implicit sparsity: {imp_sparsity:.4f}%  (expect ≈ 99.7%)")
assert 99.0 < imp_sparsity < 99.99, f"Unexpected implicit sparsity: {imp_sparsity}"

# 5) Load lại từ disk để chắc chắn artifact đã lưu đúng.
reloaded = load_npz(implicit_npz)
assert reloaded.shape == implicit_user_item_csr.shape
print("[OK] Reloaded implicit matrix shape matches in-memory.")

[OK] All required files exist.
[OK] Two distinct matrices (explicit & implicit) with matching ID maps.
[OK] No cold-start users/items remain in test_warm.
[INFO] implicit sparsity: 99.6097%  (expect ≈ 99.7%)
[OK] Reloaded implicit matrix shape matches in-memory.
